**Track A: Transporte y logística: puertas de embarque --> Caso: **nuevo** aeropuerto de Lima**

In [ ]:
#Instalación de la librería qiskit
%pip install qiskit[visualization] qiskit-aer
import qiskit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 93.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 73.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [qiskit-aer]


In [ ]:
#Modelo matemático estándar QAP10
from google.colab import files
uploaded = files.upload()

Saving qap10.mps.gz to qap10.mps (1).gz


Un aeropuerto debe asignar diez vuelos de conexión a diez puertas. Algunos vuelos comparten muchos pasajeros en tránsito; si sus puertas quedan lejos, esos pasajeros deberán recorrer más distancia.
* Flujo --> Fik: pasajeros con conexión entre los vuelos i y k
* Distancia --> Djl: distancia entre las puertas j y l  
* F: Matriz de flujo de pasajeros que tienen vuelos de conexión por número de vuelo
* D: Matriz de flujo de la distancia entre las puertas

**Fase 1: N = 3 o 4**

In [ ]:
#Con 1 repetición

In [ ]:
#BASELINE

import gzip, itertools, time, collections

# 1) Lectura de columnas -> [(fila, valor), ...]
cols = collections.OrderedDict()
seccion = None
with gzip.open('qap10.mps.gz', 'rt') as f:
    for linea in f:
        if not linea.strip() or linea.startswith('*'):
            continue
        if not linea[0].isspace():
            seccion = linea.split()[0]
            continue
        p = linea.split()
        if seccion == 'COLUMNS' and "'MARKER'" not in p:
            for r, v in zip(p[1::2], p[2::2]):
                cols.setdefault(p[0], []).append((r, float(v)))

# 2) X-> (i,j), lectura de  filas NX (sin suponer la numeración)
xpos = {}
for nombre, ent in cols.items():
    if nombre.startswith('X'):
        nx = sorted(int(r[2:]) for r, v in ent if r.startswith('NX'))
        assert len(nx) == 2
        xpos[int(nombre[1:])] = (nx[0] - 1, nx[1] - 11)   # elemento, ubicación (0..9)
assert len(xpos) == 100 and len(set(xpos.values())) == 100
inv = {ij: u for u, ij in xpos.items()}

# 3) Cada Y une dos X; su coeficiente en NOBJ es F_ik*D_jl
C = {}
for nombre, ent in cols.items():
    if nombre.startswith('Y'):
        us = sorted({int(r[r.index('A') + 1:]) for r, v in ent if r.startswith('C')})
        assert len(us) == 2
        C[tuple(us)] = sum(v for r, v in ent if r == 'NOBJ')

# Revisiones de consistencia
assert len(C) == 4050
assert all(xpos[a][0] != xpos[b][0] and xpos[a][1] != xpos[b][1] for a, b in C)
print("pares:", len(C), "| con costo > 0:", sum(v > 0 for v in C.values()))

# 4) Costo de una asignación y baseline por enumeración
def costo(perm):
    N = len(perm)
    return sum(C.get(tuple(sorted((inv[(i, perm[i])], inv[(k, perm[k])]))), 0)
               for i in range(N) for k in range(i + 1, N))

def baseline(N):
    t0 = time.time()
    mejor, mejor_p = float('inf'), None
    for p in itertools.permutations(range(N)):
        c = costo(p)
        if c < mejor:
            mejor, mejor_p = c, p
    return mejor, mejor_p, time.time() - t0

for N in (3, 4):
    c, p, t = baseline(N)
    print(f"N={N}: costo={c}, asignación={p}, tiempo={t:.4f}s")

pares: 4050 | con costo > 0: 2610
N=3: costo=24.0, asignación=(0, 1, 2), tiempo=0.0000s
N=4: costo=32.0, asignación=(1, 2, 3, 0), tiempo=0.0001s


In [ ]:
#QISKIT

#Preparación matemática
import numpy as np

def construir_qubo(N, lam):
    idx = lambda i, j: i * N + j
    n = N * N
    Q = np.zeros((n, n)); const = 0.0
    # Costo: cada par de decisiones (x_ij, x_kl) con su precio de C
    for (a, b), v in C.items():
        (i, j), (k, l) = xpos[a], xpos[b]
        if max(i, j, k, l) < N and v:
            p, q = sorted((idx(i, j), idx(k, l)))
            Q[p, q] += v
    # Penalizaciones: (sum - 1)^2 por cada fila y cada columna
    for g in range(N):
        for grupo in ([idx(g, j) for j in range(N)], [idx(j, g) for j in range(N)]):
            const += lam
            for u in grupo:
                Q[u, u] -= lam
            for u, w in itertools.combinations(grupo, 2):
                Q[u, w] += 2 * lam
    return Q, const

def a_ising(Q, const):# x = (1 - Z)/2
    n = len(Q); h = np.zeros(n); J = {}; off = const
    for u in range(n):
        off += Q[u, u] / 2; h[u] -= Q[u, u] / 2
        for w in range(u + 1, n):
            if Q[u, w]:
                q = Q[u, w]
                off += q / 4; h[u] -= q / 4; h[w] -= q / 4; J[(u, w)] = q / 4
    return h, J, off

N, LAMBDA = 4, 30
Q, const = construir_qubo(N, LAMBDA)
h, J, offset = a_ising(Q, const)
print("qubits:", N * N, "| términos ZZ:", len(J), "| offset:", offset)

qubits: 16 | términos ZZ: 96 | offset: 620.0


In [ ]:
import time, numpy as np
from scipy.optimize import minimize
from qiskit import __version__ as qv, QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

def circuito_qaoa(n, h, J, reps):
    gam = ParameterVector("γ", reps); bet = ParameterVector("β", reps)
    qc = QuantumCircuit(n); qc.h(range(n))
    for r in range(reps):
        for (u, w), v in J.items():
            qc.rzz(2 * gam[r] * v, u, w)
        for u in range(n):
            if h[u]:
                qc.rz(2 * gam[r] * h[u], u)
        for u in range(n):
            qc.rx(2 * bet[r], u)
    return qc

def correr_qaoa(N, lam=30, reps=1, maxiter=50, shots=4096, seed=7):
    n = N * N
    Q, const = construir_qubo(N, lam)
    h, J, off = a_ising(Q, const)
    H = SparsePauliOp.from_sparse_list(
        [("Z", [u], h[u]) for u in range(n) if h[u]] +
        [("ZZ", [u, w], v) for (u, w), v in J.items()], num_qubits=n)
    ansatz = circuito_qaoa(n, h, J, reps)
    est = StatevectorEstimator()
    energia = lambda p: float(est.run([(ansatz, H, p)]).result()[0].data.evs)

    x0 = np.random.default_rng(seed).uniform(0, np.pi, ansatz.num_parameters)
    t0 = time.time()
    res = minimize(energia, x0, method="COBYLA", options={"maxiter": maxiter})
    qc = ansatz.copy(); qc.measure_all()
    counts = (StatevectorSampler(seed=seed).run([(qc, res.x)], shots=shots)
              .result()[0].data.meas.get_counts())
    t_q = time.time() - t0

    fact, mejor_c, mejor_p = 0, float("inf"), None
    for bits, c in counts.items():
        x = np.array([int(b) for b in bits[::-1]]).reshape(N, N)
        if (x.sum(0) == 1).all() and (x.sum(1) == 1).all():
            fact += c
            perm = tuple(int(j) for j in x.argmax(1))
            cc = costo(perm)
            if cc < mejor_c:
                mejor_c, mejor_p = cc, perm
    return dict(N=N, qubits=n, energia_pen=res.fun + off, frac_fact=fact / shots,
                costo=mejor_c, asignacion=mejor_p, tiempo=t_q)

print(f"Qiskit {qv} | λ=30 | reps=1 | maxiter=50 | shots=4096 | semilla=7\n")
print(f"{'N':>2} | {'Método':<14} | {'Costo':>6} | {'Factibles':>9} | {'Tiempo':>8} | Asignación")
for N in (3, 4):
    c, p, t = baseline(N)
    print(f"{N:>2} | {'Clásico':<14} | {c:>6.0f} | {'100%':>9} | {t:>7.4f}s | {p}")
    r = correr_qaoa(N)
    costo_txt = f"{r['costo']:.0f}" if r['asignacion'] else "ninguno"
    print(f"{N:>2} | {'QAOA (Qiskit)':<14} | {costo_txt:>6} | {r['frac_fact']:>9.1%} | {r['tiempo']:>7.1f}s | {r['asignacion']}"
          f"   (energía penalizada {r['energia_pen']:.1f})")

Qiskit 2.6.1 | λ=30 | reps=1 | maxiter=50 | shots=4096 | semilla=7

 N | Método         |  Costo | Factibles |   Tiempo | Asignación
 3 | Clásico        |     24 |      100% |  0.0001s | (0, 1, 2)
 3 | QAOA (Qiskit)  |     26 |      0.2% |     1.9s | (1, 0, 2)   (energía penalizada 205.3)
 4 | Clásico        |     32 |      100% |  0.0003s | (1, 2, 3, 0)
 4 | QAOA (Qiskit)  | ninguno |      0.0% |    36.0s | None   (energía penalizada 618.2)


QAOA midió el circuito 4096 veces y cada medición dio una combinación de 16 interruptores. Para que cuente, tiene que ser una asignación válida (un vuelo por puerta). Ninguna de las 4096 lo fue, así que no hay costo que calcular y la tabla pone ninguno

In [ ]:
# Prueba de coherencia por el costo de "ninguno" en la prueba de QAOA con N = 4
N_t, lam, reps, maxiter, shots, seed = 4, 30, 1, 50, 4096, 7
n = N_t * N_t

Q, const = construir_qubo(N_t, lam)
h_, J_, off = a_ising(Q, const)
H = SparsePauliOp.from_sparse_list(
    [("Z", [u], h_[u]) for u in range(n) if h_[u]] +
    [("ZZ", [u, w], v) for (u, w), v in J_.items()], num_qubits=n)
ansatz = circuito_qaoa(n, h_, J_, reps)
est = StatevectorEstimator()
energia = lambda p: float(est.run([(ansatz, H, p)]).result()[0].data.evs)

x0 = np.random.default_rng(seed).uniform(0, np.pi, ansatz.num_parameters)
res = minimize(energia, x0, method="COBYLA", options={"maxiter": maxiter})

qc = ansatz.copy(); qc.measure_all()
counts = StatevectorSampler(seed=seed).run([(qc, res.x)], shots=shots).result()[0].data.meas.get_counts()

# Energía promedio de las muestras, calculada aparte con el QUBO
e_muestras = 0.0
for bits, c in counts.items():
    xv = np.array([int(b) for b in bits[::-1]], dtype=float)
    e_muestras += c * (xv @ Q @ xv + const)
e_muestras /= shots

print(f"Energía que reportó el optimizador (con offset): {res.fun + off:.1f}")
print(f"Energía promedio de las muestras medidas:        {e_muestras:.1f}")
print(f"Muestras distintas: {len(counts)} de {shots}")

Energía que reportó el optimizador (con offset): 618.2
Energía promedio de las muestras medidas:        616.4
Muestras distintas: 3899 de 4096


1. El resultado demuestra que no hay un error de lectura respecto al resultado "ninguno", ya que el optimizador esperaba una energía de 618.2 y las muestras promedian 616.4. La diferencia es de aproximadamente 1.8, lo cual es consistente con el ruido estadístico de 4096 shots. Asimismo, cabe mencionar que se obtuvieron 3899 resultados distintos de 4096, lo que indica que con reps = 1 (cantidad de repeticiones) no concentró probabilidad en las asignaciones factibles. Por lo tanto, el resultado “ninguno” refleja el comportamiento del método y no un error de código.

2. Hipótesis: Más capas aumentarían el costo de simulación sin garantizar una mejoría.

**Fase 2: N = 10**

In [ ]:
#Baseline

import numpy as np, itertools, time

N10 = 10
# W[i,k,j,l] = precio de (vuelo i en puerta j) + (vuelo k en puerta l), con i < k
W = np.zeros((N10, N10, N10, N10))
for (a, b), v in C.items():
    (i, j), (k, l) = xpos[a], xpos[b]
    if i < k: W[i, k, j, l] = v
    else:     W[k, i, l, j] = v

# Recuperar F y D (hasta un factor de escala)
pares = [(i, k) for i in range(N10) for k in range(i + 1, N10)]
ref = max(pares, key=lambda t: np.count_nonzero(W[t]))
D = W[ref] / W[ref].max() * 5                       # distancias entre puertas
F = np.zeros((N10, N10))
for t in pares: F[t] = W[t].max() / 5               # flujo entre vuelos
F = F + F.T
assert all(np.allclose(F[t] * D, W[t]) for t in pares), "F*D no reproduce C"
assert np.allclose(D, D.T) and np.allclose(np.diag(D), 0)
###print("F*D reproduce todos los precios de C | D simétrica | escala: max(D) =", D.max())

# Baseline N=10 por enumeración vectorizada
def costos_bloque(P, N):
    c = np.zeros(len(P))
    for i in range(N):
        for k in range(i + 1, N):
            c += W[i, k][P[:, i], P[:, k]]
    return c

def baseline_np(N, chunk=200_000):
    t0 = time.time()
    mejor, mejor_p = np.inf, None
    it = itertools.permutations(range(N))
    while True:
        blk = list(itertools.islice(it, chunk))
        if not blk: break
        P = np.array(blk, dtype=np.int8)
        c = costos_bloque(P, N)
        m = c.argmin()
        if c[m] < mejor:
            mejor, mejor_p = float(c[m]), tuple(int(x) for x in P[m])
    return mejor, mejor_p, time.time() - t0

for N_ in (3, 4, 10):
    c, p, t = baseline_np(N_)
    print(f"N={N_}: costo={c:.0f}, asignación={p}, tiempo={t:.2f}s")

N=3: costo=24, asignación=(0, 1, 2), tiempo=0.00s
N=4: costo=32, asignación=(1, 2, 3, 0), tiempo=0.01s
N=10: costo=340, asignación=(2, 6, 3, 0, 8, 4, 5, 1, 7, 9), tiempo=11.93s


In [ ]:
#QISKIT
#modelo híbrido
from scipy.optimize import linear_sum_assignment

def Wget(i, k, j, l):
    return W[i, k, j, l] if i < k else W[k, i, l, j]

def costo_total(p):
    return sum(W[i, k, p[i], p[k]] for i in range(N10) for k in range(i + 1, N10))

def subproblema(p, S, lam=None):
    k = len(S); doors = [p[s] for s in S]; fijos = [f for f in range(N10) if f not in S]
    Q = np.zeros((k * k, k * k)); const = 0.0; idx = lambda a, b: a * k + b
    for a in range(k):
        for b in range(k):
            Q[idx(a, b), idx(a, b)] += sum(Wget(S[a], f, doors[b], p[f]) for f in fijos)
    for a in range(k):
        for c in range(a + 1, k):
            for b in range(k):
                for e in range(k):
                    if b != e:
                        Q[idx(a, b), idx(c, e)] += Wget(S[a], S[c], doors[b], doors[e])
    if lam is None:
        lam = float(np.abs(Q).max()) + 1
    for g in range(k):
        for grupo in ([idx(g, b) for b in range(k)], [idx(a, g) for a in range(k)]):
            const += lam
            for u in grupo: Q[u, u] -= lam
            for u, w in itertools.combinations(grupo, 2): Q[u, w] += 2 * lam
    return Q, const, doors

def aplicar(p, d):
    q = list(p)
    for f, pu in d.items(): q[f] = pu
    return q

def decodificar(S, doors, x):
    if (x.sum(0) == 1).all() and (x.sum(1) == 1).all():
        return {S[a]: doors[int(x[a].argmax())] for a in range(len(S))}
    return None # None si no es permutación válida

def reparar(S, doors, probs):# reparación clásica sobre las marginales
    r, c = linear_sum_assignment(-probs)
    return {S[a]: doors[b] for a, b in zip(r, c)}

def solver_exacto(Q, const, k):# control clásico:(estados, probabilidades)
    n = k * k; X = ((np.arange(2**n)[:, None] >> np.arange(n)) & 1).astype(float)
    E = np.einsum('si,ij,sj->s', X, Q, X) + const
    pr = np.zeros(2**n); pr[E.argmin()] = 1
    return X, pr

def hibrido(solver, k=4, iters=30, seed=0):
    rng = np.random.default_rng(seed)
    p = [int(v) for v in rng.permutation(N10)]
    hist = [costo_total(p)]; reparadas = 0
    for it in range(iters):
        S = sorted(int(v) for v in rng.choice(N10, k, replace=False))
        Q, const, doors = subproblema(p, S)
        X, pr = solver(Q, const, k)
        best = None
        for s in np.argsort(-pr)[:50]:# muestras más frecuentes
            if pr[s] == 0: break
            d = decodificar(S, doors, X[s].reshape(k, k))
            if d:
                q = aplicar(p, d); c = costo_total(q)
                if best is None or c < best[0]: best = (c, q)
        if best is None:  # ninguna factible -> reparación
            probs = (pr[:, None] * X).sum(0).reshape(k, k)
            q = aplicar(p, reparar(S, doors, probs)); best = (costo_total(q), q); reparadas += 1
        if best[0] < costo_total(p): p = best[1]
        hist.append(costo_total(p))
    return p, hist, reparadas

p, h, rep = hibrido(solver_exacto, iters=40)
print(f"Control clásico: {h[0]:.0f} -> {h[-1]:.0f} | reparadas: {rep} | válida: {sorted(p)==list(range(10))}")

Control clásico: 514 -> 372 | reparadas: 0 | válida: True


In [ ]:
#Qiskit
def solver_qaoa(Q, const, k, reps=1, maxiter=50, shots=2048, seed=7):
    n = k * k
    h_, J_, off = a_ising(Q, const)
    H = SparsePauliOp.from_sparse_list(
        [("Z", [u], h_[u]) for u in range(n) if h_[u]] +
        [("ZZ", [u, w], v) for (u, w), v in J_.items()], num_qubits=n)
    ansatz = circuito_qaoa(n, h_, J_, reps)
    est = StatevectorEstimator()
    energia = lambda q_: float(est.run([(ansatz, H, q_)]).result()[0].data.evs)
    rg = np.random.default_rng(seed)
    x0 = np.concatenate([rg.uniform(0, 0.1, reps), rg.uniform(0, np.pi, reps)])  #hipótesis sin probar
    res = minimize(energia, x0, method="COBYLA", options={"maxiter": maxiter})
    qc = ansatz.copy(); qc.measure_all()
    counts = StatevectorSampler(seed=seed).run([(qc, res.x)], shots=shots).result()[0].data.meas.get_counts()
    X = np.array([[int(b) for b in bits[::-1]] for bits in counts], dtype=float)
    pr = np.array(list(counts.values()), dtype=float) / shots
    return X, pr

t = time.time()
p, h, rep = hibrido(solver_qaoa, iters=8)
print(f"Híbrido QAOA: {h[0]:.0f} -> {h[-1]:.0f} | iteraciones reparadas: {rep}/8 | tiempo {time.time()-t:.0f}s")
print("asignación:", p, "| gap vs 340:", f"{100*(h[-1]-340)/340:.1f}% --> es qué tan lejos estamos del valor óptimo")

Híbrido QAOA: 514 -> 472 | iteraciones reparadas: 8/8 | tiempo 264s
asignación: [8, 6, 0, 5, 2, 7, 9, 4, 3, 1] | gap vs 340: 38.8% --> es qué tan lejos estamos del valor óptimo


Usando qiskit con N = 10 estamos 38.8% lejos del costo óptimo

En conclusión:
*  El método clásico (baseline) supera al método que utiliza qiskit al llegar a los valores óptimos especificados en el caso.
* En las instancias N = 3 y N = 4, el método clásico (baseline) supera al enfoque con qiskit. La enumeración de permutaciones obtuvo los óptimos de control (24 y 32) en menos de un milisegundo, con factibilidad del 100 %. Mientras que usando QAOA con reps = 1 capa obtuvo un costo de 26 en N = 3.

¿Por qué el resultado de qiskit no mejora el obtenido en el método clásico?
1. Con N = 4, osea 4 vuelos, solo hay 24 formas de asignarlos, y la computadora clásica las revisa todas casi al instante. No hay nada mejor que lo que ya encuentra.
2. El circuito con reps = 1 casi no aprendió. Para N = 4, la energía penalizada esperada (618.2) quedó cerca del promedio de una elección al azar (620), muy lejos del mínimo (32).
3. Obligar a cumplir las reglas lo hace más difícil. Para que cada vuelo tenga una sola puerta, se le agrega una penalización al problema. Esto lo hace más complicado de optimizar para un circuito tan simple.
4. Con 1 capa solo hay 2 parámetros que ajustar. Usar más capas haría la simulación más lenta y no asegura una mejoría (esto último no ha sido probado, es una hipótesis).



